# Protein Crosslink Topology

This notebook turns PDB/mmCIF physical crosslinks into an embedded multigraph, extracts the **crosslink-supported bridgeless cyclic core**, and measures how its canonical Yamada fingerprint changes after selected crosslinks are removed.

The core step is essential: an open protein terminus or a bridge between cyclic blocks makes the Yamada polynomial vanish. The workflow therefore extracts the 2-core, removes remaining single bridges, and repeats the reduction after every perturbation.

A changed Yamada polynomial proves that two embeddings are topologically distinct under the package convention. An unchanged polynomial is only an unchanged fingerprint; it does **not** prove isotopy.

## 1. Imports and local data

The repository includes cached coordinates for the three paper examples. When a cache file is absent, pass a four-character PDB ID to `load_crosslinked_protein` and allow download.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from knotted_graph.applications.protein import (
    FingerprintComputer,
    FingerprintSettings,
    analyze_crosslink_perturbations,
    check_repulsor_availability,
    extract_crosslink_core,
    generate_null_graphs,
)
from knotted_graph.applications.protein.visualization import (
    plot_edge_importance,
    plot_protein_graph_3d,
)
from knotted_graph.inputs import (
    build_crosslinked_protein_graph,
    load_crosslinked_protein,
)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the KnottedGraph checkout.")

PDB_CACHE = PROJECT_ROOT / "pdb-cache"
RESULTS = PROJECT_ROOT / "results" / "protein_topology" / "user_guide"
RESULTS.mkdir(parents=True, exist_ok=True)
print("project:", PROJECT_ROOT)
print("results:", RESULTS)

## 2. Parse physical crosslinks and build the spatial graph

By default the input adapter includes disulfide and non-backbone covalent links. Metal coordination can be added explicitly. Solvent coordination and ordinary adjacent peptide `C–N` links are classified separately and excluded from the default graph.

In [ ]:
protein = load_crosslinked_protein(
    PDB_CACHE / "1AOC.pdb",
    pdb_id="1AOC",
    chain_ids=["A"],
    allowed_crosslink_types={"disulfide"},
    download=False,
)
core = extract_crosslink_core(protein.graph)

print("included crosslinks:", len(protein.crosslinks))
print("excluded records:", len(protein.excluded_crosslinks))
print("full graph (nodes, edges):", (protein.graph.number_of_nodes(), protein.graph.number_of_edges()))
print("cyclic core (nodes, edges):", (core.number_of_nodes(), core.number_of_edges()))
print("core crosslink IDs:", core.graph["core_crosslink_ids"])
print("input issues:", protein.issues)

In [ ]:
figure, axis = plot_protein_graph_3d(
    protein.graph,
    output_path=RESULTS / "1AOC_A_crosslinked_graph.png",
    title="1AOC chain A: backbone and disulfide crosslinks",
)
plt.show()

## 3. A fast, real fingerprint smoke test

The complete 1AOC core can have dozens of crossings before geometric relaxation, so exact evaluation is deliberately protected by `max_crossings`. The next cell selects one physical disulfide to make a small, genuinely coordinate-derived cycle. It exercises parsing, core extraction, projection, exact Yamada evaluation, deletion, and the empty-core convention without pretending to be the full-protein result.

In [ ]:
smoke_crosslink = protein.crosslinks[0]
smoke_graph, smoke_included, smoke_issues = build_crosslinked_protein_graph(
    protein.atom_records,
    [smoke_crosslink],
    pdb_id=protein.pdb_id,
    source_format=protein.source_format,
    chain_ids=protein.chain_ids,
    backbone_atom=protein.backbone_atom,
)
computer = FingerprintComputer(
    RESULTS / "fingerprint_cache",
    settings=FingerprintSettings(
        num_rotation_samples=5,
        max_crossings=16,
        n_jobs=1,
    ),
)
smoke_analysis = analyze_crosslink_perturbations(
    smoke_graph,
    fingerprinter=computer,
    include_pairs=False,
    enumerate_all_subsets=True,
    max_exact_crosslinks=1,
)
print("crosslink:", smoke_crosslink.crosslink_id)
print("baseline polynomial:", smoke_analysis.baseline.polynomial)
print("single-edge changed:", smoke_analysis.singles[0].changed)
print("f_top:", smoke_analysis.topological_fraction)
print("R1:", smoke_analysis.robustness_r1)
print("issues:", smoke_issues)

In [ ]:
figure, axis = plot_edge_importance(
    smoke_analysis,
    output_path=RESULTS / "1AOC_single_crosslink_importance.png",
)
plt.show()

## 4. Full single/pair/subset scan

For a graph with crosslink set $E_c$:

- $X_i=1$ when deleting edge $i$ changes the canonical fingerprint;
- $f_{top}=|\{i:X_i=1\}|/|E_c|$ and $R_1=1-f_{top}$;
- abstract-conditioned topology-carrying edges remove excess embedding topology while holding connectivity fixed;
- a subset is strictly cooperative when it is carrying but no non-empty proper subset is;
- exact enumeration returns the state lattice, while the incremental retained-set search returns a proven minimum or rigorous lower bound.

No monotonicity assumption is used. Full lattices cost $2^m$ evaluations; `--conditioned-max-subset-order` and `--minimum-generator-max-retained-crosslinks` provide separately auditable bounded analyses.

In [ ]:
RUN_FULL_EXACT_SCAN = False

if RUN_FULL_EXACT_SCAN:
    full_analysis = analyze_crosslink_perturbations(
        protein.graph,
        fingerprinter=computer,
        include_pairs=True,
        enumerate_all_subsets=True,
        max_exact_crosslinks=8,
    )
    print(full_analysis.to_dict())
else:
    print(
        "Full exact scan skipped by default. First use Repulsor/smoothing to lower "
        "the projection crossing count, then set RUN_FULL_EXACT_SCAN=True."
    )

## 5. Null models

`unique_disulfide_matchings` exactly enumerates eligible non-native perfect matchings of the same intrachain disulfide endpoints and samples without replacement only when the ensemble exceeds the requested cap. This avoids duplicate rewires as pseudoreplication. `coordinate_preserving` retains the folded coordinates and is the primary biological null. The explicit `canonical_low_crossing` mode instead tests abstract rewired connectivity and must not be described as fold-preserving. Compare raw $R_1$ with `compare_robustness_to_null` and conditioned carrying-edge fractions with `compare_conditioned_topology_to_null`.

In [ ]:
null_graphs = generate_null_graphs(
    protein,
    replicates=3,
    seed=2026,
    embedding_mode="canonical_low_crossing",
)
for record in null_graphs:
    print(
        record.replicate,
        record.seed,
        len(record.crosslinks),
        record.graph.number_of_nodes(),
        record.graph.number_of_edges(),
        record.graph.graph["null_embedding_mode"],
        record.issues,
    )

## 6. Optional Repulsor preprocessing

Repulsor acts on the **bridgeless cyclic core**. Conservative pre/post-decimation keeps at least three points per edge and accepts a shortcut only when its swept triangle stays clear of non-adjacent segments. Crosslink/core vertices can move only when explicitly enabled. Deletions run after the original/relaxed fingerprints agree or an explicitly enabled swept safe-step certificate is valid. `--repulsion-fallback-only` and the null-fallback options avoid changing already-evaluable graphs. Native build, certificate, topology mismatch, and exact-cap failures remain structured data.

On Apple Silicon/macOS the driver uses the system Accelerate framework. Linux/WSL native dependencies are listed in `doc/user_guide/repulsive_layout.md`.

In [ ]:
availability = check_repulsor_availability()
print(availability)

RUN_REPULSOR = False
if RUN_REPULSOR:
    from knotted_graph.applications.protein import relax_and_analyze_crosslinks
    from knotted_graph.layout.repulsive import SolverOptions

    end_to_end_computer = FingerprintComputer(
        RESULTS / "end_to_end_cache",
        settings=FingerprintSettings(
            num_rotation_samples=32,
            max_crossings=24,
            n_jobs=-1,
        ),
    )
    repulsion_result = relax_and_analyze_crosslinks(
        protein.graph,
        RESULTS / "repulsor_1AOC",
        fingerprinter=end_to_end_computer,
        solver_options=SolverOptions(steps=20),
        allow_certificate_only=True,
        include_pairs=False,
    )
    print(repulsion_result.to_dict())
else:
    print("Repulsor run skipped; set RUN_REPULSOR=True after the pinned checkout is ready.")

## 7. Batch execution and interpretation

The frozen manifests and final commands are documented in `examples/protein_topology/README.md`. The final population profile uses `population_conditioned_recovered_v1.csv`, exact unique fold-preserving nulls, and certificate-checked fallback only for capped natural or null states.

Outputs include resumable per-sample JSON, raw and conditioned edge/subset tables, exact or bounded minimum generating sets, stable local-lasso groups, verified abstract-isomorphism groups, `dataset_statistics.json`, figures, and Repulsor certificates. The final local population run has 114/114 successful natural proteins and 398/398 successful selected unique nulls; its conditioned natural-minus-null carrying-edge fraction is -0.078484 with bootstrap 95% CI [-0.122403, -0.036190] and paired sign-flip $p=0.0002699973$. The nested 82-protein no-natural-fallback analysis preserves the effect direction and significance.

Use status, failed comparisons, crossing counts, null ensemble provenance, embedding mode, and Repulsor topology status as analysis columns—not as footnotes. Do not interpret missing or capped exact fingerprints as unchanged topology. The inference is conditional on the declared recovered exact-evaluable analytic cohort, not the whole PDB.